# Football_data_co_uk data exploration and cleaning

## Inroduction

The purpose of this article is to explore the **Football-Data.co.uk** dataset.By going through its structure, understanding its features and undercover some essential insights about it, the article tends to make the reader familiar with the **Football-Data.co.uk**.

Something that is very important is that while the dataset is being explained, it will also be cleaned and prepared for future analysis and modeling.

As a process of work the article will follow a standard pipeline which inlcudes:
- Get a first look at the data and understand it as it is possible
- Check for missing values
- Check for duplicates (This also refers to invalid and unlogical values)
- Fix the data types (This mainly refers to the fixing of the highly incorrect and incompatible data types, e.g. int value in a str type column)
- Optimize the memory usage (This is about casting certain columns type to ones with sizes, which leads to the reduction of memory usage across the dataset!)
- Fix any text fields problems, including wrong encoding chars, white spaces etc!
- Remove useless columns
- Rename/Reorder the columns(if needed, for better clarity)
- Check for unlogical and invalid values such as outliers
- Make some little analyses, visualizations and form assumptions (This is something which will be more detailly observed during the EDA project phase!)
- Store it into a file

This is the pipeline which will be mainly followed during the data exploration and cleaning, and of course there could be many deviations due to data specifications!

Now lets begin the exploration.

## Football-Data.co.uk

Now the first thing that I want to explain is the dataset itself.What is the dataset about: \
If we look at the title of the project, it says football **betting** analyses.The word that I want you to give special attentian to, is the word **betting**.Well, this is what the dataset is all about: **betting**!More specifically the dataset provides data about historical betting **coefficients** of different widely-known and reliable **bookmakers**.

**What is a bookmaker**?
The job of the bookmakers, especialy the ones which analyze football data and predict future matches, is to give the most realistic and professional predictions, based on what they have as statistics and historical data!The predictions they made are represented under the form of football betting **odds** of also knows as football **coefficients**.

**What are football odds**?
The odds give a very simple and straight-forward information: What are the probabilities that a specific team in a specific match, will win, loss or make a draw with its opponent.These predictions are spread into many different scenarios of differet outcomes in a football match, but at its very basic, just provides a prediction of win, loss and draw.That's it, nothign more complicated.Well, yes, but it is not so simple as it looks like!And the project itself shows that!To be honest, a football match is something which is really hard to be predicted, mostly because the football is a game of surprise - you never know what is going to happen, and thats why it is so interesting!So in order for these predictions to really be reliable and have a constistent behaviour, they really should have been constructed based on enormous amount of historical football data, which includes players statistics, teams forms, mathces, event-level data and even external factors such as stadioms, match attendence, home or away match and many more.And all this data should be combined and processed in a very professional anmd comprehensive way, which is refined down to the smallest detail!

**How the Football-Data will help the project and how will be used**?
Well, actually, the football-data.co.uk is the one of the best and most reliable football betting odds priveder in the whole world, as it is getting its odds from all kinds of different platforms all around the world.Visit the platform: [Football-Data.co.uk](https://www.football-data.co.uk/).Moreover the dataset ensures that the provided odds are alwats up to date, and it give you the most recent odds data for almost every competition, league, season and team in the world, going all way back to years 1993/1994.It is really really professional site!So on the question how the dataset will be used for the purposes of the project, the answer should already be clear to you!Of course that the dataset will be used to get historical bettings odds!As you should already know, one of the project main aims is to try to create suffuciently reliable statistical model for predictions of football matches!So how this dataset will come in play, is that when the model is finished, it will be calibrated with the bettings odds provided of this dataset!

**I want to specify that the dataset also contains other statistics, which could be used in other aspects of the project, except for the calibration of the bettings odds!**

So I think that I have explained, mainly, what the dataset is about, how would it be useful for the prpject and also it should have helped you to understand what the betting odds are and how they are calculated!

So lets begin work! \
I will start by, importing the required libraries for the article!

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from football_betting_analysis.config import FOOTBALL_DATA_INTERIM_PATH, START_SEASON, END_SEASON

from football_betting_analysis.data.text_cleaning import clean_text_values
from football_betting_analysis.data.save_data_into_file import save_data
from football_betting_analysis.data.data_cleaning import convert_string_to_datetime, optimize_dataframe_memory
from football_betting_analysis.data.team_match_validation import validate_team_matches

Now lets load the dataset, which is at the following location: `data/raw/football_data_co_uk/`

In [4]:
football_data = pd.read_csv('../../data/raw/football_data_co_uk/matches.csv')

In [5]:
football_data

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,1XBD,1XBA,B365H,B365D,B365A,BFH,BFD,BFA,season,league
0,SP1,2014-08-23,NaN,Almeria,Espanol,1,1,D,0,0,...,NaN,NaN,2.55,3.30,2.70,NaN,NaN,NaN,2014/2015,SP1
1,SP1,2014-08-23,NaN,Granada,La Coruna,2,1,H,0,1,...,NaN,NaN,1.95,3.20,4.20,NaN,NaN,NaN,2014/2015,SP1
2,SP1,2014-08-23,NaN,Malaga,Ath Bilbao,1,0,H,1,0,...,NaN,NaN,2.80,3.30,2.45,NaN,NaN,NaN,2014/2015,SP1
3,SP1,2014-08-23,NaN,Sevilla,Valencia,1,1,D,1,0,...,NaN,NaN,2.05,3.50,3.50,NaN,NaN,NaN,2014/2015,SP1
4,SP1,2014-08-24,NaN,Barcelona,Elche,3,0,H,1,0,...,NaN,NaN,1.10,9.00,21.00,NaN,NaN,NaN,2014/2015,SP1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4175,SP1,2025-05-24,20:00,Getafe,Celta,1,2,A,1,1,...,3.76,1.89,4.10,3.75,1.85,4.20,3.60,1.90,2024/2025,SP1
4176,SP1,2025-05-24,20:00,Vallecano,Mallorca,0,0,D,0,0,...,4.27,6.84,1.53,4.20,6.25,1.55,4.00,6.50,2024/2025,SP1
4177,SP1,2025-05-25,13:00,Girona,Ath Madrid,0,4,A,0,0,...,3.89,2.02,3.80,3.75,1.91,4.10,3.75,1.87,2024/2025,SP1
4178,SP1,2025-05-25,15:15,Villarreal,Sevilla,4,2,H,3,1,...,5.04,6.58,1.48,5.00,6.00,1.47,5.00,6.50,2024/2025,SP1


As we can in the dataset, most of the columns are with vague names.This is just because, that is the way that **Football-Data.co.uk** privides its data.But this is not a problem because I will explain each of the columns and then I will rename them, so that it is visual and clear what they are about!

So, lets explain each of the columns:

#### **Key to results data:**
- Div = League Division
- Date = Match Date (dd/mm/yy)
- Time = Time of match kick off
- HomeTeam = Home Team
- AwayTeam = Away Team
- FTHG and HG = Full Time Home Team Goals
- FTAG and AG = Full Time Away Team Goals
- FTR and Res = Full Time Result (H=Home Win, D=Draw, A=Away Win)
- HTHG = Half Time Home Team Goals
- HTAG = Half Time Away Team Goals
- HTR = Half Time Result (H=Home Win, D=Draw, A=Away Win)

#### **Match Statistics:**
- Attendance = Crowd Attendance
- Referee = Match Referee
- HS = Home Team Shots
- AS = Away Team Shots
- HST = Home Team Shots on Target
- AST = Away Team Shots on Target
- HHW = Home Team Hit Woodwork
- AHW = Away Team Hit Woodwork
- HC = Home Team Corners
- AC = Away Team Corners
- HF = Home Team Fouls Committed
- AF = Away Team Fouls Committed
- HFKC = Home Team Free Kicks Conceded
- AFKC = Away Team Free Kicks Conceded
- HO = Home Team Offsides
- AO = Away Team Offsides
- HY = Home Team Yellow Cards
- AY = Away Team Yellow Cards
- HR = Home Team Red Cards
- AR = Away Team Red Cards

I want to note that Free Kicks Conceeded includes fouls, offsides and any other offense commmitted and will always be equal to or higher than the number of fouls. Fouls make up the vast majority of Free Kicks Conceded. Free Kicks Conceded are shown when specific data on Fouls are not available!


#### **Betting odds:**
The following key to betting odds data is described below. These are for pre-closing odds. For the closing odds, as below but with an additional "C" character following the bookmaker abbreviation/Max/Avg (e.g. B365CH = closing Bet365 home win odds).

##### 1XBet bookmaker:
- 1XBH = 1XBet home win odds
- 1XBD = 1XBet draw odds
- 1XBA = 1XBet away win odds

##### Bet365 bookmaker:
- B365H = Bet365 home win odds
- B365D = Bet365 draw odds
- B365A = Bet365 away win odds

##### Betfair bookmaker:
- BFH = Betfair home win odds
- BFD = Betfair draw odds
- BFA = Betfair away win odds

#### Additional info:
- season = the season when the match was played
- league = the league code of the Spanish La Liga(SP1) - Spain first devision

#### This is the dataset features
The number of columns should be 42, so lets check it:

In [6]:
football_data.shape

(4180, 42)

In [7]:
len(football_data.columns)

42

Now, what I am thinking is that, it would be best if I rename the columns names from now, so that it is more easy to understand what we are looking at.We should do it at some moment anyways, so yes!

So lest define an dict, which will hold as keys the current features names and as values their corresponding changed names - The new features names!

In [8]:
COLUMN_RENAME_MAP = {
    # GENERAL INFO
    "Div": "league_division",
    "Date": "datetime",
    "Time": "kickoff_time",
    "HomeTeam": "h_title",
    "AwayTeam": "a_title",

    # FULL TIME RESULTS
    "FTHG": "home_goals_full",
    "FTAG": "away_goals_full",
    "FTR": "result_full",

    # HALF TIME RESULTS
    "HTHG": "home_goals_half",
    "HTAG": "away_goals_half",
    "HTR": "result_half",

    # MATCH META
    "Attendance": "attendance",
    "Referee": "referee",

    # SHOTS
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_shots_on_target",
    "AST": "away_shots_on_target",

    # WOODWORK
    "HHW": "home_hit_woodwork",
    "AHW": "away_hit_woodwork",
    
    # CORNERS
    "HC": "home_corners",
    "AC": "away_corners",

    # FOULS
    "HF": "home_fouls",
    "AF": "away_fouls",

    # FREE KICKS CONCEDED
    "HFKC": "home_free_kicks_conceded",
    "AFKC": "away_free_kicks_conceded",

    # OFFSIDES
    "HO": "home_offsides",
    "AO": "away_offsides",

    # CARDS
    "HY": "home_yellow_cards",
    "AY": "away_yellow_cards",
    "HR": "home_red_cards",
    "AR": "away_red_cards",

    # ODDS - 1XBET
    "1XBH": "odds_1xbet_home",
    "1XBD": "odds_1xbet_draw",
    "1XBA": "odds_1xbet_away",

    # ODDS - BET365
    "B365H": "odds_bet365_home",
    "B365D": "odds_bet365_draw",
    "B365A": "odds_bet365_away",

    # ODDS - BETFAIR
    "BFH": "odds_betfair_home",
    "BFD": "odds_betfair_draw",
    "BFA": "odds_betfair_away",
    
    "league": "league_code"
}

The only thing that I am not renaming is the season feature name, because it it clear! So to ensure that everything else is ok, lets check the len of the dict.It should be **41**!

In [9]:
len(COLUMN_RENAME_MAP)

41

Something which I almost forgot to do is to make a copy of the raw data, so that the raw data to stay untouched!This is very important!

In [10]:
football_data_interim = football_data.copy()

In [11]:
football_data_interim

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,1XBD,1XBA,B365H,B365D,B365A,BFH,BFD,BFA,season,league
0,SP1,2014-08-23,NaN,Almeria,Espanol,1,1,D,0,0,...,NaN,NaN,2.55,3.30,2.70,NaN,NaN,NaN,2014/2015,SP1
1,SP1,2014-08-23,NaN,Granada,La Coruna,2,1,H,0,1,...,NaN,NaN,1.95,3.20,4.20,NaN,NaN,NaN,2014/2015,SP1
2,SP1,2014-08-23,NaN,Malaga,Ath Bilbao,1,0,H,1,0,...,NaN,NaN,2.80,3.30,2.45,NaN,NaN,NaN,2014/2015,SP1
3,SP1,2014-08-23,NaN,Sevilla,Valencia,1,1,D,1,0,...,NaN,NaN,2.05,3.50,3.50,NaN,NaN,NaN,2014/2015,SP1
4,SP1,2014-08-24,NaN,Barcelona,Elche,3,0,H,1,0,...,NaN,NaN,1.10,9.00,21.00,NaN,NaN,NaN,2014/2015,SP1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4175,SP1,2025-05-24,20:00,Getafe,Celta,1,2,A,1,1,...,3.76,1.89,4.10,3.75,1.85,4.20,3.60,1.90,2024/2025,SP1
4176,SP1,2025-05-24,20:00,Vallecano,Mallorca,0,0,D,0,0,...,4.27,6.84,1.53,4.20,6.25,1.55,4.00,6.50,2024/2025,SP1
4177,SP1,2025-05-25,13:00,Girona,Ath Madrid,0,4,A,0,0,...,3.89,2.02,3.80,3.75,1.91,4.10,3.75,1.87,2024/2025,SP1
4178,SP1,2025-05-25,15:15,Villarreal,Sevilla,4,2,H,3,1,...,5.04,6.58,1.48,5.00,6.00,1.47,5.00,6.50,2024/2025,SP1


## Renaming Columns:
Now lets rename the columns:

In [12]:
football_data_interim = football_data_interim.rename(columns=COLUMN_RENAME_MAP)

In [13]:
football_data_interim.dtypes

league_division                 str
datetime                        str
kickoff_time                    str
h_title                         str
a_title                         str
home_goals_full               int64
away_goals_full               int64
result_full                     str
home_goals_half               int64
away_goals_half               int64
result_half                     str
attendance                  float64
referee                     float64
home_shots                    int64
away_shots                    int64
home_shots_on_target          int64
away_shots_on_target          int64
home_hit_woodwork           float64
away_hit_woodwork           float64
home_corners                  int64
away_corners                  int64
home_fouls                    int64
away_fouls                    int64
home_free_kicks_conceded    float64
away_free_kicks_conceded    float64
home_offsides               float64
away_offsides               float64
home_yellow_cards           

> I want to specially pay attention to the **datetime, h_title and a_title columns**.These columns were renamed in that way because this is the way that the other datasets, that I have cleaned by now, use as names for these columns.So in order to keep the things consistent and more importanly to make them suitable for marges, I have made the datasets to use these exact same names for these specific unique columns!

Ok, now it is much better!

# Missing values:
Now something which is important and I want to see is, **are there any missing values in the dataset!**

In [14]:
football_data_interim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 42 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   league_division           4180 non-null   str    
 1   datetime                  4180 non-null   str    
 2   kickoff_time              2280 non-null   str    
 3   h_title                   4180 non-null   str    
 4   a_title                   4180 non-null   str    
 5   home_goals_full           4180 non-null   int64  
 6   away_goals_full           4180 non-null   int64  
 7   result_full               4180 non-null   str    
 8   home_goals_half           4180 non-null   int64  
 9   away_goals_half           4180 non-null   int64  
 10  result_half               4180 non-null   str    
 11  attendance                0 non-null      float64
 12  referee                   0 non-null      float64
 13  home_shots                4180 non-null   int64  
 14  away_shots         

Ok we can see that there are many of them.

Lets see excatly which of them are missing:

In [15]:
missing_values_columns = football_data_interim.isna().any()

missing_values_columns[missing_values_columns.values == True]

kickoff_time                True
attendance                  True
referee                     True
home_hit_woodwork           True
away_hit_woodwork           True
home_free_kicks_conceded    True
away_free_kicks_conceded    True
home_offsides               True
away_offsides               True
odds_1xbet_home             True
odds_1xbet_draw             True
odds_1xbet_away             True
odds_betfair_home           True
odds_betfair_draw           True
odds_betfair_away           True
dtype: bool

In [16]:
missing_values_columns = football_data_interim.isna().all()
missing_values_columns[missing_values_columns.values == True]

attendance                  True
referee                     True
home_hit_woodwork           True
away_hit_woodwork           True
home_free_kicks_conceded    True
away_free_kicks_conceded    True
home_offsides               True
away_offsides               True
dtype: bool

There columns contain only missing values, which is not good, actually. \
Well in such situations I have no choice but removing the entire columns.

In matter of fact, none of these is something which is special and cannot be gathered from another place.I have a lot of data sources which I can guarantee that contains all of these specific data!So this is none of a problem to the dataset and to the purposes of the project at all, so I can freely remove them!

The thing which is important is that the dataset provides the most important data, that actually was required to get from this dataset specifically, and this is the odds data.Yes, **the 1xbet and the betfair odds** are limited but I have full coverage of the **bet365 odds**, which, by the way, is the most famous, trust-worthy and professional bookmaker in the world.So I think that this would be more than enough!

Now enough talking, lets remove these columns:

In [17]:
full_missing_cols = missing_values_columns[missing_values_columns.values == True].index

football_data_interim = football_data_interim.drop(columns=full_missing_cols)

In [18]:
any(football_data_interim.isna().all())

False

In [19]:
len(football_data_interim.columns)

34

Ok we remove those, but what are we goign to do with the others:

In [20]:
missing_values_columns = football_data_interim.isna().any()

missing_values_columns[missing_values_columns.values == True]

kickoff_time         True
odds_1xbet_home      True
odds_1xbet_draw      True
odds_1xbet_away      True
odds_betfair_home    True
odds_betfair_draw    True
odds_betfair_away    True
dtype: bool

Well, what would I do with features which are extremely limited based on the size of the database.

The 1xbet and the betfair odds contain 380 records each, it is just useless. \
The kickoff time is half the size of the dataset, but it doesn't provide any meaningful information that could be used somewhere.

### What do I mean by **meaningful information**?
This is something that I want to specifically pay attention to and explain!
> By meaningful information, I mean either, a data that could be useful for the objectives of the project, which includes football betting/matches/teams performance analyses or a data which in someway could be used in some way in the statistical predictive models and the improvement of their predictions!One thing which also falls into the definition of a meaningful information is a data which is required in the dataset, so that the dataset could be related to any of the other relative datasets.By relate, I mean **merge**.And also this can be interpreted as a data which is unique for the specific dataset and without it, the dataset neither losses its ability to be related with othet datasets or its main unique stats ---- This is what I mean when I say meaningful(useful) information - To be clear from now on!

So when I have stated what a meaningful information mean lets remove these **un**meaningful information:

In [21]:
partially_missing_cols = missing_values_columns[missing_values_columns.values == True].index

football_data_interim = football_data_interim.drop(columns=partially_missing_cols)

In [22]:
any(football_data_interim.isna().any())

False

In [23]:
len(football_data_interim.columns)

27

Ok we are done with the missing values!

## Teams, Season and matches validation: 
Now lets check the if there are **380** matches per season and also if there are 38 matches for each of the teams in each of the seasons!

In [24]:
seasons = [season for season in range(START_SEASON, END_SEASON)]

for season in seasons:
    current_season_matches = football_data_interim[football_data_interim['season'].str[:4] == str(season)]
    print(f'Season: {season}/{season + 1}, Matches: {len(current_season_matches)}')

Season: 2014/2015, Matches: 380
Season: 2015/2016, Matches: 380
Season: 2016/2017, Matches: 380
Season: 2017/2018, Matches: 380
Season: 2018/2019, Matches: 380
Season: 2019/2020, Matches: 380
Season: 2020/2021, Matches: 380
Season: 2021/2022, Matches: 380
Season: 2022/2023, Matches: 380
Season: 2023/2024, Matches: 380
Season: 2024/2025, Matches: 380


Ok for the seasons everything is correct!

Now next thing I want to check is: Has every team played 38 matches in a single season? \
I will do this by using the `validate_team_matches` function which is implemented in the following location [team_match_validation](../../src/football_betting_analysis/data/team_match_validation.py)

The function validates if every team in each of the seasons have played a total of 38 matches!And returns a dict with the amount of valid teams in each of the seasons:A valid team is considered a team which has played a total of 38 matches in the specific season!

So lets check that using the function:

In [25]:
valid_teams = validate_team_matches(football_data_interim, 'h_title', 'a_title')

In [26]:
valid_teams

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

Well everything is perfect!


## Duplicates
Now lets check if there are any duplicates!

The duplication criteria that I will define is the following: **home_team + away_team + date**.It will be very bad if there is more than one value of this criteria!

So lets check it:

In [27]:
dupes = football_data_interim.duplicated(
    subset=['datetime', 'h_title', 'a_title']
)

dupes.any()

np.False_

Ok this is very good.There are no duplicates!

## Fixing the data types:

Now lets see the data types of the dataset and fix something if needed:

In [28]:
football_data_interim.dtypes

league_division             str
datetime                    str
h_title                     str
a_title                     str
home_goals_full           int64
away_goals_full           int64
result_full                 str
home_goals_half           int64
away_goals_half           int64
result_half                 str
home_shots                int64
away_shots                int64
home_shots_on_target      int64
away_shots_on_target      int64
home_corners              int64
away_corners              int64
home_fouls                int64
away_fouls                int64
home_yellow_cards         int64
away_yellow_cards         int64
home_red_cards            int64
away_red_cards            int64
odds_bet365_home        float64
odds_bet365_draw        float64
odds_bet365_away        float64
season                      str
league_code                 str
dtype: object

Ok now the only thing which is surely wrong is the datetime column, which is with type string, so lets fix that.

But before that lets check that all of the datetimes are with the same format, to prevent any problems in the casting.From what I see the format is: `YYYY/MM/DD`

I will both check and convert the columns with the functions: `convert_string_to_datetime`! \
So lets do it:

In [29]:
football_data_interim['datetime'] = convert_string_to_datetime(football_data_interim['datetime'], '%Y-%m-%d')

In [30]:
football_data_interim.dtypes

league_division                    str
datetime                datetime64[us]
h_title                            str
a_title                            str
home_goals_full                  int64
away_goals_full                  int64
result_full                        str
home_goals_half                  int64
away_goals_half                  int64
result_half                        str
home_shots                       int64
away_shots                       int64
home_shots_on_target             int64
away_shots_on_target             int64
home_corners                     int64
away_corners                     int64
home_fouls                       int64
away_fouls                       int64
home_yellow_cards                int64
away_yellow_cards                int64
home_red_cards                   int64
away_red_cards                   int64
odds_bet365_home               float64
odds_bet365_draw               float64
odds_bet365_away               float64
season                   

Ok this was fixed!

## Memory usage:
Now lets optimize the memory usage of the dataset using the `optimize_dataframe_memory`!

In [31]:
football_data_interim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   league_division       4180 non-null   str           
 1   datetime              4180 non-null   datetime64[us]
 2   h_title               4180 non-null   str           
 3   a_title               4180 non-null   str           
 4   home_goals_full       4180 non-null   int64         
 5   away_goals_full       4180 non-null   int64         
 6   result_full           4180 non-null   str           
 7   home_goals_half       4180 non-null   int64         
 8   away_goals_half       4180 non-null   int64         
 9   result_half           4180 non-null   str           
 10  home_shots            4180 non-null   int64         
 11  away_shots            4180 non-null   int64         
 12  home_shots_on_target  4180 non-null   int64         
 13  away_shots_on_target  4180 no

In [32]:
football_data_interim = optimize_dataframe_memory(football_data_interim)

Initial Memory Usage: 0.99 MB
Final Memory Usage: 0.17 MB
Memory Reduction: 82.55%

Optimized Data Types:
league_division               category
datetime                datetime64[us]
h_title                       category
a_title                       category
home_goals_full                   int8
away_goals_full                   int8
result_full                   category
home_goals_half                   int8
away_goals_half                   int8
result_half                   category
home_shots                        int8
away_shots                        int8
home_shots_on_target              int8
away_shots_on_target              int8
home_corners                      int8
away_corners                      int8
home_fouls                        int8
away_fouls                        int8
home_yellow_cards                 int8
away_yellow_cards                 int8
home_red_cards                    int8
away_red_cards                    int8
odds_bet365_home               float

In [33]:
football_data_interim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   league_division       4180 non-null   category      
 1   datetime              4180 non-null   datetime64[us]
 2   h_title               4180 non-null   category      
 3   a_title               4180 non-null   category      
 4   home_goals_full       4180 non-null   int8          
 5   away_goals_full       4180 non-null   int8          
 6   result_full           4180 non-null   category      
 7   home_goals_half       4180 non-null   int8          
 8   away_goals_half       4180 non-null   int8          
 9   result_half           4180 non-null   category      
 10  home_shots            4180 non-null   int8          
 11  away_shots            4180 non-null   int8          
 12  home_shots_on_target  4180 non-null   int8          
 13  away_shots_on_target  4180 no

Ok we can see the huge reducement of memory that we have made!

## Removing useless columns:

Now lets remove some useless columns that I have noticed from earlier.

This columns would be **the league_division and the league_code**, just because they falls into the fall under the common denominator: **unmeaningful data**!They will not be used for any merges, just because all of the data is from the La Liga, so there would not be any deviations on that and also are not prociding any unique or model helpful data!

So lets remove them:

In [34]:
football_data_interim = football_data_interim.drop(columns=['league_division', 'league_code'])

In [35]:
football_data_interim.columns

Index(['datetime', 'h_title', 'a_title', 'home_goals_full', 'away_goals_full',
       'result_full', 'home_goals_half', 'away_goals_half', 'result_half',
       'home_shots', 'away_shots', 'home_shots_on_target',
       'away_shots_on_target', 'home_corners', 'away_corners', 'home_fouls',
       'away_fouls', 'home_yellow_cards', 'away_yellow_cards',
       'home_red_cards', 'away_red_cards', 'odds_bet365_home',
       'odds_bet365_draw', 'odds_bet365_away', 'season'],
      dtype='str')

## Reordering Columns:
Ok now lets reorder the dataset so that it is properly structured:

In [36]:
football_data_interim = football_data_interim[
    [
       'season', 'datetime', 'h_title', 'a_title', 'home_goals_full', 'away_goals_full',
       'result_full', 'home_goals_half', 'away_goals_half', 'result_half',
       'home_shots', 'away_shots', 'home_shots_on_target',
       'away_shots_on_target', 'home_corners', 'away_corners', 'home_fouls',
       'away_fouls', 'home_yellow_cards', 'away_yellow_cards',
       'home_red_cards', 'away_red_cards', 'odds_bet365_home',
       'odds_bet365_draw', 'odds_bet365_away'
    ]
]

In [37]:
football_data_interim

,season,datetime,h_title,a_title,home_goals_full,away_goals_full,result_full,home_goals_half,away_goals_half,result_half,...,away_corners,home_fouls,away_fouls,home_yellow_cards,away_yellow_cards,home_red_cards,away_red_cards,odds_bet365_home,odds_bet365_draw,odds_bet365_away
0,2014/2015,2014-08-23,Almeria,Espanol,1,1,D,0,0,D,...,7,8,9,3,2,0,1,2.55,3.30,2.70
1,2014/2015,2014-08-23,Granada,La Coruna,2,1,H,0,1,A,...,3,13,26,1,2,0,0,1.95,3.20,4.20
2,2014/2015,2014-08-23,Malaga,Ath Bilbao,1,0,H,1,0,H,...,4,13,9,3,3,2,0,2.80,3.30,2.45
3,2014/2015,2014-08-23,Sevilla,Valencia,1,1,D,1,0,H,...,3,23,8,4,2,0,1,2.05,3.50,3.50
4,2014/2015,2014-08-24,Barcelona,Elche,3,0,H,1,0,H,...,1,11,13,0,1,1,0,1.10,9.00,21.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4175,2024/2025,2025-05-24,Getafe,Celta,1,2,A,1,1,D,...,6,13,10,3,4,1,0,4.10,3.75,1.85
4176,2024/2025,2025-05-24,Vallecano,Mallorca,0,0,D,0,0,D,...,2,9,14,2,2,0,0,1.53,4.20,6.25
4177,2024/2025,2025-05-25,Girona,Ath Madrid,0,4,A,0,0,D,...,8,13,11,0,1,0,0,3.80,3.75,1.91
4178,2024/2025,2025-05-25,Villarreal,Sevilla,4,2,H,3,1,H,...,5,8,14,0,1,0,0,1.48,5.00,6.00


Ok, now it is better!

# Dealing with text features:

One thing that I have forgotten to do is to fix any inconsistencies of the text features in the dataset!

So lets fix that by using the func: `clean_text_values`:

In [38]:
text_cols = football_data_interim.select_dtypes(include=['category']).columns

football_data_interim[text_cols] = football_data_interim[text_cols].apply(
    clean_text_values
)

In [39]:
football_data_interim[text_cols]

,season,h_title,a_title,result_full,result_half
0,2014/2015,Almeria,Espanol,D,D
1,2014/2015,Granada,La Coruna,H,A
2,2014/2015,Malaga,Ath Bilbao,H,H
3,2014/2015,Sevilla,Valencia,D,H
4,2014/2015,Barcelona,Elche,H,H
...,...,...,...,...,...
4175,2024/2025,Getafe,Celta,A,D
4176,2024/2025,Vallecano,Mallorca,D,D
4177,2024/2025,Girona,Ath Madrid,A,D
4178,2024/2025,Villarreal,Sevilla,H,H


## Saving dataset into a file:

Lastly, lets save the dataset into the `data/interim/` directory.

Something which should be done before that, is to create a `football_data_co_uk` folder inside the `data/interim/` dir so that I can save the cleaned dataset into its specific folder:

In [40]:
# The FOOTBALL_DATA_INTERIM_PATH is a constant defined in the scr/config.py script which is used to store global variables and important constants among the project!
# force working directory to project root
PROJECT_ROOT = Path().resolve().parent.parent

file_path = PROJECT_ROOT / FOOTBALL_DATA_INTERIM_PATH
file_path.mkdir(parents=True, exist_ok=True)

Ok now as we have a folder, lets save the dataset into the folder.To do that I am going to use the `save_data` function, which is implementated at the following path: `src/football_betting_analysis/save_data_into_file`.This function will simply store the dataset into the specifed path location, into a **parquet** file!

I am using **parquet** files, because they keep metadata about the data which helps the data to preserve everything it has as a features structure and data types.It is much more optimized than many more files types and it is very easy to work with!

So lets save the file:

In [41]:
PROJECT_ROOT = Path().resolve().parent.parent
file_path = PROJECT_ROOT / FOOTBALL_DATA_INTERIM_PATH / 'interim_matches_v1.parquet'

save_data(data=football_data_interim, file_path=file_path)

The file has already been created and it contains the exact data as the original dataset!
